# 05 - Analise de Materiais
> Interpretacao de resultados, protocolos de ensaio e laudos tecnicos

**Modulo:** `12_enfitec_servicos` | **Feito com:** [Claude](https://claude.ai) (Anthropic)

---

In [ ]:
import os
from dotenv import load_dotenv
import anthropic

load_dotenv()
client = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))
HAIKU  = 'claude-haiku-4-5-20251001'
SONNET = 'claude-sonnet-4-5'

def ask(prompt, system=None, model=HAIKU, max_tokens=1024):
    kw = dict(model=model, max_tokens=max_tokens,
              messages=[{'role':'user','content':prompt}])
    if system: kw['system'] = system
    return client.messages.create(**kw).content[0].text

print('Pronto!')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Configuracao padrao dos graficos
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
print('NumPy e Matplotlib prontos!')

---
## Interpretando Difratogramas de Raios-X

A difracao de raios-X (DRX / XRD) e uma tecnica fundamental para identificar
fases cristalinas em materiais. Vamos gerar dados sinteticos de um difratograma
e usar Claude para interpretar os resultados.

In [ ]:
# --- Gerando dados sinteticos de XRD (aluminio FCC) ---
np.random.seed(42)

two_theta = np.linspace(20, 90, 1000)

# Picos caracteristicos do aluminio (FCC)
# (111) ~38.5, (200) ~44.7, (220) ~65.1, (311) ~78.2, (222) ~82.4
al_peaks = [
    (38.47, 100, 0.3),   # 2theta, intensidade relativa, largura
    (44.74,  47, 0.35),
    (65.13,  22, 0.4),
    (78.23,  24, 0.45),
    (82.43,   8, 0.5),
]

intensity = np.random.normal(2, 0.5, len(two_theta))  # ruido de fundo
intensity = np.clip(intensity, 0, None)

for pos, amp, width in al_peaks:
    peak = amp * np.exp(-0.5 * ((two_theta - pos) / width) ** 2)
    intensity += peak

# Plotar o difratograma
fig, ax = plt.subplots()
ax.plot(two_theta, intensity, 'b-', linewidth=0.8)
ax.set_xlabel('2-Theta (graus)')
ax.set_ylabel('Intensidade (u.a.)')
ax.set_title('Difratograma de Raios-X - Amostra Desconhecida')
ax.set_xlim(20, 90)

# Anotar picos
for pos, amp, _ in al_peaks:
    ax.annotate(f'{pos:.1f}', xy=(pos, amp+3), fontsize=8, ha='center')

plt.tight_layout()
plt.show()
print(f'Numero de pontos: {len(two_theta)}')
print(f'Picos detectados em 2-theta: {[p[0] for p in al_peaks]}')

In [ ]:
# --- Formatando dados para enviar ao Claude ---
peaks_text = "Picos detectados no difratograma de raios-X:\n"
peaks_text += "2-Theta (graus) | Intensidade Relativa (%)\n"
peaks_text += "-" * 45 + "\n"
for pos, amp, _ in al_peaks:
    peaks_text += f"  {pos:>6.2f}         |  {amp:>6.1f}\n"

print(peaks_text)
print("---")
print("Dados formatados e prontos para analise!")

In [ ]:
# --- Pedindo ao Claude para identificar fases e calcular parametro de rede ---
system_xrd = """Voce e um especialista em caracterizacao de materiais por
difracao de raios-X (XRD). Analise os dados de picos fornecidos e:
1. Identifique a(s) fase(s) cristalina(s) presente(s)
2. Determine a estrutura cristalina (FCC, BCC, HCP, etc.)
3. Calcule o parametro de rede usando a lei de Bragg (lambda = 1.5406 A, Cu K-alpha)
4. Indexe os planos cristalograficos (hkl) de cada pico
Mostre os calculos passo a passo."""

prompt_xrd = f"""Analise os seguintes dados de difracao de raios-X:

{peaks_text}

Comprimento de onda utilizado: Cu K-alpha = 1.5406 Angstroms
Identifique o material e calcule o parametro de rede."""

resp_xrd = ask(prompt_xrd, system=system_xrd, model=SONNET, max_tokens=2048)
print(resp_xrd)

In [ ]:
# --- Calculo de verificacao: parametro de rede do Al ---
wavelength = 1.5406  # Angstroms (Cu K-alpha)

# Indices de Miller para FCC: (111), (200), (220), (311), (222)
hkl_indices = [(1,1,1), (2,0,0), (2,2,0), (3,1,1), (2,2,2)]
peak_positions = [38.47, 44.74, 65.13, 78.23, 82.43]

print("Verificacao do parametro de rede (a):")
print("=" * 55)
a_values = []
for (h, k, l), two_th in zip(hkl_indices, peak_positions):
    theta_rad = np.radians(two_th / 2)
    d = wavelength / (2 * np.sin(theta_rad))
    a = d * np.sqrt(h**2 + k**2 + l**2)
    a_values.append(a)
    print(f"  ({h}{k}{l})  2theta={two_th:6.2f}  d={d:.4f} A  a={a:.4f} A")

a_mean = np.mean(a_values)
a_std = np.std(a_values)
print(f"\nParametro de rede medio: a = {a_mean:.4f} +/- {a_std:.4f} A")
print(f"Valor tabelado para Al: a = 4.0495 A")
print(f"Erro relativo: {abs(a_mean - 4.0495)/4.0495*100:.2f}%")

---
## Analise de Curvas de Tracao

O ensaio de tracao e o teste mecanico mais comum para caracterizar materiais.
Vamos gerar uma curva tensao-deformacao sintetica e extrair propriedades mecanicas.

In [ ]:
# --- Gerando curva tensao-deformacao sintetica (aco 1045) ---
np.random.seed(123)

# Regiao elastica
E_modulus = 205  # GPa (modulo de Young)
yield_strength = 530  # MPa
eps_yield = yield_strength / (E_modulus * 1000)  # deformacao no escoamento

eps_elastic = np.linspace(0, eps_yield, 100)
sigma_elastic = E_modulus * 1000 * eps_elastic  # MPa

# Regiao plastica (modelo de Hollomon: sigma = K * eps_p^n)
K = 900  # MPa (coeficiente de resistencia)
n = 0.15  # expoente de encruamento
uts = 680  # MPa (resistencia maxima)

eps_plastic = np.linspace(eps_yield, 0.25, 300)
eps_p = eps_plastic - eps_yield
sigma_plastic = yield_strength + K * (eps_p ** n)

# Limitar pela UTS e adicionar necking
idx_uts = np.argmin(np.abs(sigma_plastic - uts))
sigma_plastic[idx_uts:] = uts * np.exp(-2 * (eps_plastic[idx_uts:] - eps_plastic[idx_uts]))

# Combinar regioes
strain = np.concatenate([eps_elastic, eps_plastic[1:]])
stress = np.concatenate([sigma_elastic, sigma_plastic[1:]])

# Adicionar ruido
stress += np.random.normal(0, 2, len(stress))
stress = np.clip(stress, 0, None)

# Plotar a curva
fig, ax = plt.subplots()
ax.plot(strain * 100, stress, 'b-', linewidth=1.2)
ax.set_xlabel('Deformacao (%)')
ax.set_ylabel('Tensao (MPa)')
ax.set_title('Curva Tensao-Deformacao - Aco 1045')
ax.axhline(y=yield_strength, color='r', linestyle='--', alpha=0.5, label=f'Escoamento ~{yield_strength} MPa')
ax.axhline(y=uts, color='g', linestyle='--', alpha=0.5, label=f'UTS ~{uts} MPa')
ax.legend()
ax.set_xlim(0, 25)
ax.set_ylim(0, 800)
plt.tight_layout()
plt.show()

print(f'Pontos de dados: {len(strain)}')
print(f'Deformacao maxima: {strain[-1]*100:.1f}%')

In [ ]:
# --- Extraindo propriedades e pedindo analise ao Claude ---

# Resumo dos dados para o Claude
# Amostragem a cada N pontos para nao exceder contexto
step = 10
data_summary = "Dados do ensaio de tracao (amostragem):\n"
data_summary += "Deformacao (%) | Tensao (MPa)\n"
data_summary += "-" * 35 + "\n"
for i in range(0, len(strain), step):
    data_summary += f"  {strain[i]*100:>8.3f}     | {stress[i]:>8.1f}\n"

system_mech = """Voce e um engenheiro de materiais especialista em ensaios mecanicos.
Analise a curva tensao-deformacao fornecida e determine:
1. Modulo de Young (regiao elastica linear)
2. Limite de escoamento (offset 0.2%)
3. Limite de resistencia a tracao (UTS)
4. Alongamento total
5. Tenacidade (area sob a curva)
6. Classificacao do material (fragil, ductil, etc.)
Mostre os calculos quando possivel."""

prompt_mech = f"""Analise os seguintes dados de ensaio de tracao:

{data_summary}

Determine as propriedades mecanicas do material."""

resp_mech = ask(prompt_mech, system=system_mech, model=SONNET, max_tokens=2048)
print(resp_mech)

In [ ]:
# --- Calculo programatico das propriedades mecanicas ---

# 1. Modulo de Young: regressao linear na regiao elastica
mask_elastic = strain < eps_yield * 0.8  # regiao seguramente elastica
coeffs = np.polyfit(strain[mask_elastic], stress[mask_elastic], 1)
E_calc = coeffs[0] / 1000  # converter para GPa

# 2. Limite de escoamento (offset 0.2%)
offset_line = coeffs[0] * (strain - 0.002) + coeffs[1]
diff = stress - offset_line
idx_yield = np.where((diff[:-1] > 0) & (diff[1:] <= 0))[0]
yield_calc = stress[idx_yield[0]] if len(idx_yield) > 0 else yield_strength

# 3. UTS
uts_calc = np.max(stress)
idx_uts_calc = np.argmax(stress)

# 4. Alongamento total
elong_total = strain[-1] * 100

# 5. Tenacidade (area sob a curva por trapezio)
tenacidade = np.trapz(stress, strain)  # MPa (= MJ/m^3)

print("Propriedades Mecanicas Calculadas:")
print("=" * 45)
print(f"  Modulo de Young:     {E_calc:.1f} GPa (nominal: {E_modulus} GPa)")
print(f"  Limite Escoamento:   {yield_calc:.0f} MPa (nominal: {yield_strength} MPa)")
print(f"  UTS:                 {uts_calc:.0f} MPa (nominal: {uts} MPa)")
print(f"  Alongamento total:   {elong_total:.1f}%")
print(f"  Tenacidade:          {tenacidade:.1f} MJ/m^3")

---
## Espectroscopia: FTIR e UV-Vis

Tecnicas espectroscopicas sao essenciais para identificar grupos funcionais
(FTIR) e transicoes eletronicas (UV-Vis). Vamos gerar espectros sinteticos
e pedir ao Claude para interpretar os picos.

In [ ]:
# --- Gerando espectro FTIR sintetico (polietileno) ---
np.random.seed(55)

wavenumber = np.linspace(4000, 400, 2000)

# Picos caracteristicos do polietileno (PE)
ftir_peaks = [
    (2920, 0.85, 30),   # C-H stretching assimetrico
    (2850, 0.75, 25),   # C-H stretching simetrico
    (1470, 0.45, 20),   # C-H bending (tesoura)
    (1370, 0.20, 15),   # C-H bending (wagging)
    (730,  0.35, 18),   # C-H rocking
    (720,  0.30, 15),   # C-H rocking (dubleto)
]

# Baseline (transmitancia)
transmittance = np.ones(len(wavenumber)) * 0.95
transmittance += np.random.normal(0, 0.005, len(wavenumber))

# Adicionar picos de absorcao (inverter para transmitancia)
for pos, depth, width in ftir_peaks:
    peak = depth * np.exp(-0.5 * ((wavenumber - pos) / width) ** 2)
    transmittance -= peak

transmittance = np.clip(transmittance, 0.01, 1.0)

# Plotar espectro FTIR
fig, ax = plt.subplots()
ax.plot(wavenumber, transmittance * 100, 'b-', linewidth=0.8)
ax.set_xlabel('Numero de Onda (cm$^{-1}$)')
ax.set_ylabel('Transmitancia (%)')
ax.set_title('Espectro FTIR - Amostra Polimerica')
ax.invert_xaxis()  # convencao FTIR
ax.set_ylim(0, 105)

for pos, depth, _ in ftir_peaks:
    ax.annotate(f'{pos}', xy=(pos, (0.95 - depth) * 100 - 3),
                fontsize=7, ha='center', color='red')

plt.tight_layout()
plt.show()

In [ ]:
# --- Gerando espectro UV-Vis sintetico (corante organico) ---
np.random.seed(77)

wavelength_uv = np.linspace(200, 800, 1000)

# Picos de absorcao: transicoes pi->pi* e n->pi*
uv_peaks = [
    (280, 0.8, 15),   # Transicao pi->pi* (anel aromatico)
    (320, 0.3, 12),   # Transicao n->pi* (carbonila)
    (450, 1.2, 40),   # Banda de absorcao principal (cromoforo)
    (550, 0.4, 30),   # Ombro da banda visivel
]

absorbance = np.zeros(len(wavelength_uv))
absorbance += np.random.normal(0, 0.005, len(wavelength_uv))

for pos, amp, width in uv_peaks:
    peak = amp * np.exp(-0.5 * ((wavelength_uv - pos) / width) ** 2)
    absorbance += peak

absorbance = np.clip(absorbance, 0, None)

# Plotar UV-Vis
fig, ax = plt.subplots()
ax.plot(wavelength_uv, absorbance, 'purple', linewidth=1.0)
ax.fill_between(wavelength_uv, absorbance, alpha=0.1, color='purple')
ax.set_xlabel('Comprimento de Onda (nm)')
ax.set_ylabel('Absorbancia (u.a.)')
ax.set_title('Espectro UV-Vis - Corante Organico')

for pos, amp, _ in uv_peaks:
    ax.annotate(f'{pos} nm', xy=(pos, amp + 0.05), fontsize=8,
                ha='center', color='darkred')

plt.tight_layout()
plt.show()

In [ ]:
# --- Analise espectroscopica com Claude ---
system_spectro = """Voce e um espectroscopista experiente.
Analise os dados espectrais fornecidos e:
1. Identifique cada pico/banda com seu grupo funcional ou transicao
2. Sugira a composicao provavel do material
3. Indique se ha contaminantes ou impurezas
4. Recomende analises complementares se necessario."""

# FTIR
ftir_text = "Espectro FTIR - Picos de absorcao detectados:\n"
ftir_text += "Numero de onda (cm-1) | Profundidade relativa\n"
ftir_text += "-" * 45 + "\n"
for pos, depth, _ in ftir_peaks:
    ftir_text += f"      {pos:>5d}            |    {depth:.2f}\n"

# UV-Vis
uv_text = "\nEspectro UV-Vis - Bandas de absorcao:\n"
uv_text += "Lambda max (nm) | Absorbancia\n"
uv_text += "-" * 35 + "\n"
for pos, amp, _ in uv_peaks:
    uv_text += f"     {pos:>4d}        |    {amp:.2f}\n"

prompt_spectro = f"""Analise os seguintes espectros de uma amostra desconhecida:

{ftir_text}
{uv_text}

Identifique os grupos funcionais, transicoes eletronicas e
sugira a composicao provavel do material."""

resp_spectro = ask(prompt_spectro, system=system_spectro, model=SONNET, max_tokens=2048)
print(resp_spectro)

---
## Protocolos de Ensaio

Protocolos bem definidos garantem reprodutibilidade e confiabilidade dos resultados.
Vamos usar Claude para gerar protocolos completos de ensaio para diferentes
tecnicas de caracterizacao.

In [ ]:
# --- Protocolo para analise termica (DSC, TGA, DMA) ---
system_protocol = """Voce e um especialista em caracterizacao de materiais
responsavel por elaborar protocolos de ensaio conforme normas tecnicas.
Gere protocolos detalhados incluindo:
- Preparacao de amostras (dimensoes, massa, condicionamento)
- Parametros de medicao (taxa de aquecimento, atmosfera, faixa de temperatura)
- Configuracao do equipamento
- Calibracao necessaria
- Cuidados e precaucoes
- Normas de referencia (ASTM, ISO)
- Criterios de aceitacao
Use formato de tabela quando apropriado."""

prompt_dsc = """Elabore um protocolo completo de ensaio para as seguintes
tecnicas de analise termica de um polimero termoplastico (nylon 6,6):

1. DSC (Calorimetria Diferencial de Varredura)
   - Objetivo: Determinar Tg, Tm, Tc, grau de cristalinidade

2. TGA (Analise Termogravimetrica)
   - Objetivo: Determinar estabilidade termica, teor de carga mineral

3. DMA (Analise Dinamico-Mecanica)
   - Objetivo: Determinar modulos E' e E'', tan delta, Tg dinamica

Para cada tecnica, inclua preparacao de amostra, parametros de medicao,
configuracoes do equipamento e normas aplicaveis."""

resp_protocol = ask(prompt_dsc, system=system_protocol, model=SONNET, max_tokens=4096)
print(resp_protocol)

In [ ]:
# --- Protocolo para ensaios mecanicos ---
prompt_mech_protocol = """Elabore um protocolo detalhado para os seguintes ensaios
mecanicos em corpos de prova de aco AISI 4340:

1. Ensaio de tracao (ASTM E8)
2. Ensaio de impacto Charpy (ASTM E23)
3. Ensaio de dureza Rockwell C (ASTM E18)
4. Ensaio de fadiga rotativa (ASTM E466)

Inclua:
- Geometria e dimensoes dos corpos de prova
- Velocidade de ensaio / frequencia
- Temperatura de ensaio
- Numero minimo de repeticoes
- Criterios de validade do ensaio"""

resp_mech_protocol = ask(prompt_mech_protocol, system=system_protocol,
                         model=SONNET, max_tokens=4096)
print(resp_mech_protocol)

---
## Selecao de Materiais

A selecao de materiais combina requisitos de projeto com propriedades dos materiais.
A metodologia de Ashby utiliza graficos de propriedade vs. propriedade para
identificar candidatos otimos.

In [ ]:
# --- Banco de dados simplificado de materiais ---
materiais = {
    'Aco 1020':       {'E': 205, 'sigma_y': 350, 'rho': 7.85, 'T_max': 400, 'custo': 0.8},
    'Aco Inox 316':   {'E': 193, 'sigma_y': 290, 'rho': 8.00, 'T_max': 800, 'custo': 4.0},
    'Al 6061-T6':     {'E': 69,  'sigma_y': 276, 'rho': 2.70, 'T_max': 150, 'custo': 2.5},
    'Ti-6Al-4V':      {'E': 114, 'sigma_y': 880, 'rho': 4.43, 'T_max': 400, 'custo': 25.0},
    'Nylon 6,6':      {'E': 3.0, 'sigma_y': 70,  'rho': 1.14, 'T_max': 80,  'custo': 3.5},
    'PEEK':           {'E': 4.1, 'sigma_y': 100, 'rho': 1.30, 'T_max': 250, 'custo': 90.0},
    'Epoxi + FC':     {'E': 70,  'sigma_y': 600, 'rho': 1.60, 'T_max': 180, 'custo': 50.0},
    'Alumina (Al2O3)':{'E': 380, 'sigma_y': 300, 'rho': 3.90, 'T_max': 1500,'custo': 8.0},
    'Cobre C11000':   {'E': 117, 'sigma_y': 70,  'rho': 8.94, 'T_max': 200, 'custo': 6.0},
    'Mg AZ31B':       {'E': 45,  'sigma_y': 200, 'rho': 1.77, 'T_max': 120, 'custo': 5.0},
}

# Legenda: E (GPa), sigma_y (MPa), rho (g/cm3), T_max (C), custo ($/kg)

print(f"{'Material':<18} {'E(GPa)':>7} {'Sy(MPa)':>8} {'rho':>5} {'Tmax':>5} {'$/kg':>5}")
print("-" * 55)
for nome, props in materiais.items():
    print(f"{nome:<18} {props['E']:>7.0f} {props['sigma_y']:>8.0f} "
          f"{props['rho']:>5.2f} {props['T_max']:>5d} {props['custo']:>5.1f}")

In [ ]:
# --- Grafico tipo Ashby: Resistencia especifica vs. Rigidez especifica ---
fig, ax = plt.subplots(figsize=(10, 8))

colors = plt.cm.Set3(np.linspace(0, 1, len(materiais)))

for i, (nome, props) in enumerate(materiais.items()):
    # Propriedades especificas (dividido pela densidade)
    rigidez_esp = props['E'] / props['rho']      # GPa/(g/cm3)
    resist_esp = props['sigma_y'] / props['rho']  # MPa/(g/cm3)

    ax.scatter(rigidez_esp, resist_esp, s=props['custo'] * 15 + 50,
              c=[colors[i]], edgecolors='black', linewidth=0.8, zorder=3)
    ax.annotate(nome, (rigidez_esp, resist_esp),
                textcoords='offset points', xytext=(8, 5),
                fontsize=8)

ax.set_xlabel('Rigidez Especifica - E/rho (GPa*cm$^3$/g)')
ax.set_ylabel('Resistencia Especifica - Sy/rho (MPa*cm$^3$/g)')
ax.set_title('Grafico de Ashby: Resistencia vs. Rigidez Especificas\n'
             '(tamanho do marcador proporcional ao custo)')
ax.set_xscale('log')
ax.set_yscale('log')
ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

In [ ]:
# --- Selecao de materiais com Claude ---
system_selecao = """Voce e um engenheiro de materiais especialista em selecao
de materiais usando a metodologia de Ashby. Analise os requisitos da aplicacao
e recomende materiais adequados considerando:
- Indices de merito (ex: E/rho para rigidez leve, sigma_y/rho para resistencia leve)
- Restricoes eliminatorias (temperatura, resistencia quimica)
- Custo relativo
- Fabricabilidade
Justifique cada recomendacao."""

# Formatar banco de dados para o prompt
db_text = "Banco de dados de materiais disponiveis:\n"
db_text += f"{'Material':<18} {'E(GPa)':>7} {'Sy(MPa)':>8} {'rho(g/cm3)':>10} {'Tmax(C)':>7} {'$/kg':>5}\n"
db_text += "-" * 60 + "\n"
for nome, p in materiais.items():
    db_text += f"{nome:<18} {p['E']:>7.0f} {p['sigma_y']:>8.0f} {p['rho']:>10.2f} {p['T_max']:>7d} {p['custo']:>5.1f}\n"

prompt_selecao = f"""Selecione o melhor material para a seguinte aplicacao:

APLICACAO: Braco robotico industrial
REQUISITOS:
- Temperatura de operacao: -20C a 80C
- Carga maxima: 50 kg no extremo do braco (comprimento 1.2 m)
- Rigidez: deflexao maxima de 2 mm sob carga
- Peso: o mais leve possivel (robo colaborativo)
- Resistencia quimica: contato com oleos lubrificantes e solventes leves
- Producao: 500 unidades/ano
- Orcamento: custo moderado

{db_text}

Recomende os 3 melhores candidatos, justificando com indices de merito."""

resp_selecao = ask(prompt_selecao, system=system_selecao, model=SONNET, max_tokens=3000)
print(resp_selecao)

---
## Laudo Tecnico Automatizado

Um laudo tecnico e o documento que formaliza os resultados de ensaios.
Vamos automatizar a geracao de laudos a partir de dados brutos e observacoes.

In [ ]:
# --- Dados brutos para o laudo ---
dados_laudo = {
    'cliente': 'Metalurgica ABC Ltda.',
    'material': 'Aco AISI 4340 temperado e revenido',
    'lote': 'L-2024-0847',
    'norma_ref': 'ASTM A829',
    'data_ensaio': '2024-11-15',
    'operador': 'Eng. Carlos Santos - CREA 123456',
    'ensaios': {
        'tracao': {
            'norma': 'ASTM E8/E8M',
            'cp_diametro_mm': 12.5,
            'cp_gauge_mm': 50,
            'resultados': [
                {'cp': 1, 'sigma_y': 1120, 'uts': 1240, 'along': 12.3, 'ra': 45.2},
                {'cp': 2, 'sigma_y': 1105, 'uts': 1225, 'along': 11.8, 'ra': 43.8},
                {'cp': 3, 'sigma_y': 1135, 'uts': 1250, 'along': 12.7, 'ra': 46.1},
            ],
        },
        'dureza': {
            'norma': 'ASTM E18',
            'escala': 'Rockwell C (HRC)',
            'resultados': [38, 39, 37, 38, 40, 39, 38, 37, 39, 38],
        },
        'impacto': {
            'norma': 'ASTM E23',
            'tipo': 'Charpy V',
            'temperatura': -40,
            'resultados_J': [28, 25, 30],
        },
        'metalografia': {
            'ataque': 'Nital 3%',
            'microestrutura': 'Martensita revenida com carbonetos finos dispersos',
            'tamanho_grao': 'ASTM 7-8 (conforme ASTM E112)',
            'inclusoes': 'Tipo A (sulfetos) fino, serie fina, nivel 1 (ASTM E45)',
        },
    },
    'requisitos_minimos': {
        'sigma_y_min': 1000,  # MPa
        'uts_min': 1150,      # MPa
        'along_min': 10,      # %
        'ra_min': 35,         # %
        'hrc_min': 35,
        'hrc_max': 42,
        'impacto_min': 20,    # J a -40C
    },
}

print("Dados do ensaio carregados com sucesso!")
print(f"Cliente: {dados_laudo['cliente']}")
print(f"Material: {dados_laudo['material']}")
print(f"Lote: {dados_laudo['lote']}")

In [ ]:
# --- Gerando o laudo tecnico com Claude ---
import json

system_laudo = """Voce e um engenheiro de materiais senior responsavel pela
emissao de laudos tecnicos em um laboratorio acreditado. Gere um laudo
tecnico completo e profissional no seguinte formato:

1. OBJETIVO
2. IDENTIFICACAO DA AMOSTRA
3. NORMAS DE REFERENCIA
4. METODOLOGIA E EQUIPAMENTOS
5. RESULTADOS
   - Tabelas com resultados individuais e medias
   - Comparacao com requisitos minimos (APROVADO/REPROVADO)
6. ANALISE METALOGRAFICA
7. DISCUSSAO
   - Correlacao entre propriedades
   - Conformidade com especificacoes
8. CONCLUSAO
   - Parecer final: APROVADO ou REPROVADO
9. OBSERVACOES E RECOMENDACOES

Use linguagem tecnica formal. Inclua calculos de media e desvio padrao.
O laudo deve ser autocontido e compreensivel."""

prompt_laudo = f"""Gere um laudo tecnico completo baseado nos seguintes dados:

{json.dumps(dados_laudo, indent=2, ensure_ascii=True)}

Compare todos os resultados com os requisitos minimos e emita parecer final."""

resp_laudo = ask(prompt_laudo, system=system_laudo, model=SONNET, max_tokens=4096)
print(resp_laudo)

In [ ]:
# --- Verificacao automatica dos resultados ---

print("VERIFICACAO AUTOMATICA DOS RESULTADOS")
print("=" * 60)

reqs = dados_laudo['requisitos_minimos']
ensaios = dados_laudo['ensaios']
aprovado_geral = True

# Tracao
print("\n--- Ensaio de Tracao ---")
for prop, label, req_key in [
    ('sigma_y', 'Limite Escoamento (MPa)', 'sigma_y_min'),
    ('uts', 'Limite Resistencia (MPa)', 'uts_min'),
    ('along', 'Alongamento (%)', 'along_min'),
    ('ra', 'Reducao de Area (%)', 'ra_min'),
]:
    vals = [r[prop] for r in ensaios['tracao']['resultados']]
    media = np.mean(vals)
    desvio = np.std(vals, ddof=1)
    req_val = reqs[req_key]
    ok = media >= req_val
    status = 'OK' if ok else 'REPROVADO'
    if not ok: aprovado_geral = False
    print(f"  {label}: {media:.1f} +/- {desvio:.1f} (min: {req_val}) [{status}]")

# Dureza
print("\n--- Dureza ---")
vals_hrc = ensaios['dureza']['resultados']
media_hrc = np.mean(vals_hrc)
desvio_hrc = np.std(vals_hrc, ddof=1)
ok_hrc = reqs['hrc_min'] <= media_hrc <= reqs['hrc_max']
if not ok_hrc: aprovado_geral = False
status_hrc = 'OK' if ok_hrc else 'REPROVADO'
print(f"  HRC: {media_hrc:.1f} +/- {desvio_hrc:.1f} "
      f"(faixa: {reqs['hrc_min']}-{reqs['hrc_max']}) [{status_hrc}]")

# Impacto
print("\n--- Impacto Charpy ---")
vals_imp = ensaios['impacto']['resultados_J']
media_imp = np.mean(vals_imp)
desvio_imp = np.std(vals_imp, ddof=1)
ok_imp = media_imp >= reqs['impacto_min']
if not ok_imp: aprovado_geral = False
status_imp = 'OK' if ok_imp else 'REPROVADO'
print(f"  Energia a {ensaios['impacto']['temperatura']}C: "
      f"{media_imp:.1f} +/- {desvio_imp:.1f} J (min: {reqs['impacto_min']} J) [{status_imp}]")

# Parecer final
print("\n" + "=" * 60)
parecer = 'APROVADO' if aprovado_geral else 'REPROVADO'
print(f"PARECER FINAL: *** {parecer} ***")
print("=" * 60)

In [ ]:
# --- Salvar laudo em arquivo texto ---
nome_arquivo = f"laudo_{dados_laudo['lote'].replace('-','_')}.txt"

with open(nome_arquivo, 'w', encoding='utf-8') as f:
    f.write("LAUDO TECNICO DE CARACTERIZACAO DE MATERIAIS\n")
    f.write("=" * 60 + "\n\n")
    f.write(f"Cliente: {dados_laudo['cliente']}\n")
    f.write(f"Material: {dados_laudo['material']}\n")
    f.write(f"Lote: {dados_laudo['lote']}\n")
    f.write(f"Data: {dados_laudo['data_ensaio']}\n")
    f.write(f"Operador: {dados_laudo['operador']}\n")
    f.write("\n" + "-" * 60 + "\n\n")
    f.write(resp_laudo)
    f.write("\n\n" + "=" * 60 + "\n")
    f.write("Documento gerado automaticamente com auxilio de IA.\n")

print(f"Laudo salvo em: {nome_arquivo}")
print(f"Tamanho: {os.path.getsize(nome_arquivo)} bytes")

---
## Exercicios

### Exercicio 1: Analise de DRX de mistura
Gere dados sinteticos de XRD contendo **duas fases** (ex: aluminio + oxido de aluminio).
Envie ao Claude e peca para identificar ambas as fases e estimar a proporcao relativa.

### Exercicio 2: Curva S-N de fadiga
Crie dados sinteticos de uma curva S-N (tensao vs. numero de ciclos ate a falha).
Peca ao Claude para determinar o limite de fadiga e ajustar o modelo de Basquin.

### Exercicio 3: Espectro Raman
Gere um espectro Raman sintetico de grafeno (bandas D, G, 2D) e peca ao Claude
para avaliar a qualidade do grafeno com base na razao I(D)/I(G) e I(2D)/I(G).

### Exercicio 4: Selecao multiobjetivo
Defina uma aplicacao com requisitos conflitantes (ex: alta resistencia + baixo peso +
baixo custo) e use Claude para fazer uma analise de Pareto com os materiais do banco.

### Exercicio 5: Laudo de falha
Crie um cenario de falha em servico (fratura de eixo, por exemplo) com dados de
fractografia, analise quimica e mecanica. Peca ao Claude para gerar um laudo
de analise de falha com identificacao do mecanismo e causa raiz.

---
## Proximos Passos

Neste notebook exploramos como Claude pode auxiliar na analise de materiais:

1. **Interpretacao de XRD** - Identificacao de fases e calculo de parametros cristalograficos
2. **Ensaios mecanicos** - Extracao automatica de propriedades de curvas tensao-deformacao
3. **Espectroscopia** - Identificacao de grupos funcionais e transicoes eletronicas
4. **Protocolos** - Geracao de procedimentos padronizados conforme normas tecnicas
5. **Selecao de materiais** - Recomendacao baseada em indices de merito e restricoes
6. **Laudos tecnicos** - Automatizacao de relatorios formais com parecer tecnico

### Para aprofundar:
- Integrar com bases de dados reais (ICDD para XRD, NIST para espectroscopia)
- Adicionar processamento de imagens de microscopia (MEV, MO)
- Implementar fluxos de trabalho completos (da amostra ao laudo)
- Conectar com sistemas LIMS (Laboratory Information Management System)
- Explorar analise estatistica avancada (Weibull para fadiga, DOE para otimizacao)

---
*Notebook do modulo 12 - Enfitec Servicos*